In [25]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [26]:
def llm(prompt):
    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=prompt
    )
    return response.output_text

In [27]:
llm("hey, what's up?")

"Not much, just here to help with any questions or topics you'd like to discuss. How's your day going so far?"

In [28]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

You've just discovered the course, however, I need a bit more information to help you. What course are you referring to? Is it an online course, a university course, or a workshop? Additionally, what is the current date and the course's start and end dates? With more context, I'll do my best to guide you through the process of joining the course.


In [29]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [30]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [31]:
answer = llm(prompt)
print(answer)

You can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [32]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [33]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [34]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [35]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [36]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [37]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [38]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [39]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [40]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [41]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [42]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [43]:
search_results = search("")
search_results

[]

In [44]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [45]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [46]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [47]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [59]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:


In [50]:
response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=prompt
    )

In [56]:
response.output_text

"It seems like you're interested in joining a course, but I don't have enough information about the course or its enrollment period. Could you please provide more context or details about the course, such as its duration, start date, and enrollment deadline? That way, I can give you a more accurate answer."

In [60]:
response.usage

ResponseUsage(input_tokens=50, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=63, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=113)

In [61]:
input_price = 0.59 / 1_000_000
output_price = 0.79 / 1_000_000

cost = (response.usage.input_tokens  * input_price + 
        response.usage.output_tokens * output_price )
print(cost)

7.927e-05


In [63]:
prompt

'Question:\nI just discovered the course. Can I join now?\n\nContext:'

In [62]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=message_history
)

In [65]:
def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [66]:
def rag(query, model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [77]:
answer=rag("how do I set up a DLT project??")
answer

"To set up a new DLT project, especially when loading data from the cloud, you can start by using the following command on the command line:\n\n```bash\ndlt init filesystem duckdb\n```\n\nThis command initializes a DLT project with DuckDB as the database system, which is suitable for managing and processing data.\n\nFor more detailed instructions and guidance on setting up your DLT project, you can refer to the tutorial provided on the DLT Hub website at [dlthub.com/docs/tutorial/filesystem](https://dlthub.com/docs/tutorial/filesystem). This resource should offer step-by-step directions to help you get started with your project.\n\nAdditionally, if you're looking to integrate your project with other databases like ClickHouse or StarRocks, note that DLT currently does not have native connectors for these systems. However, the DLT community is open to contributions, so if you're interested in adding support for these databases, you might consider contributing to the project.\n\nIf you ha